# 데모데이 프로젝트 — 스스로 배우는 셀프케어 코치

> 테마: **교육 & 학습** · LangGraph 교육 에이전트

## Step 1. 에이전트 설계

**이름**: 내 몸·마음 읽기 코치 (Self-Care Socratic Coach)

**목적 (어떤 문제를 해결하나)**
건강·식단 정보는 넘치는데, 사람들은 "답"만 받고 **스스로 판단하는 힘은 안 길러져요.**
이 코치는 답을 대신 주지 않고, **소크라테스식 질문**으로 사용자가 자기 몸·마음 신호를
스스로 읽고 판단하는 법을 **"배우게"** 합니다. (물고기를 주지 않고 낚시를 가르치는 교육 에이전트)

**핵심 기능 (5가지)**
1. **진단** — 사용자가 이미 아는 개념인지 판단해 가르칠 개념을 고름 (진도 메모리 기반)
2. **소크라테스식 질문** — 답 대신 "뭐가 빠진 것 같아요?"로 스스로 생각하게 함
3. **힌트 루프** — 못 맞히면 단계별 힌트(최대 2회), 그래도 모르면 부드럽게 정답 (조건 엣지 + 루프)
4. **진도 메모리** — 배운 개념을 state에 기록 → 다음엔 아는 건 건너뛰고 새 개념으로
5. **안전 가드** — 위험 신호(가슴통증·자해 등) 감지 시 코치 빠지고 전문가/1393 안내

> **LLM 적용**: `evaluate`(의미 판정)·`empathy`·`closing`은 LLM(gpt-4o-mini)로 자연 생성/판정하고,
> 나머지는 규칙기반입니다. `OPENAI_API_KEY`가 없으면 전부 규칙기반으로 자동 폴백 → 키 없이도 실행돼요.

**그래프 구조 (노드·엣지)**

```text
START
  → safety_guard ─(위험)→ [종료] 전문가·1393 안내
                 └(정상)→ empathy → diagnose
                                     ├ (다 배움) ───────────────→ closing → END
                                     └ (가르칠 것 있음) → socratic_q
                                                            │  [유저 답 대기: interrupt_before]
                                                            ▼
                                                          evaluate
                                                            ├ (정답) → praise ──→ closing → END
                                                            └ (오답·모름) → hint
                                                                            ├ (힌트 남음) →⟲ socratic_q
                                                                            └ (힌트 소진) → closing → END
```

- **분기**(conditional edges) 3곳: 앎/모름 · 채점(정답/오답) · 힌트 소진
- **루프**: hint → socratic_q (최대 2회)
- **유저 답 대기**: socratic_q 뒤에서 멈춤 = `interrupt_before` + checkpointer


## Step 2. 기초 구축 (LangGraph)

아래 순서로 자족 실행됩니다: **State → 개념 데이터 → 노드 → 그래프 → 데모**.
(설치: `pip install langgraph langchain-openai`)


In [1]:
# State — 그래프가 들고 다니는 공유 데이터 (커스텀)
"""그래프가 들고 다니는 상태 (그래프설계.md §1).

messages는 add_messages 리듀서로 대화를 누적하고, 나머지는 커스텀 TypedDict 필드.
"""
from typing import Annotated, TypedDict

from langgraph.graph.message import add_messages


class CoachState(TypedDict, total=False):
    # add_messages 리듀서: 노드가 return한 messages를 '덮어쓰기'가 아니라 '누적'.
    # ("ai", "텍스트") 튜플도 알아서 메시지 객체로 변환해준다.
    messages: Annotated[list, add_messages]
    today_input: str      # 이번 턴 사용자 입력 (예: "점심 라면+김밥")
    target_concept: str   # 이번에 가르칠 개념 key (concepts.py)
    learned: dict         # 진도 메모리 {concept_key: 숙련도} — 이미 앎/모름 판단
    hint_count: int       # 현재 개념 힌트 몇 번 줬나 (루프 종료용, max=HINT_MAX)
    user_answer: str      # 소크라테스 질문에 대한 유저 답
    verdict: str          # "correct" | "wrong" | "unknown"
    risk_flag: bool       # 의학 위험신호 감지 여부
    stage: str            # 현재 노드 (디버그·재개용)


HINT_MAX = 2  # 힌트 2번까지, 3번째도 못 맞히면 정답 알려주고 넘어감 (그래프설계.md §4)


In [2]:
# 가르칠 개념 (식단 도메인 예시) — 순수 데이터
"""식단 도메인에서 가르칠 개념 (그래프설계.md §5). criteria.txt 근거.

순수 데이터 — LLM/LangGraph 무관. diagnose 노드가 여기서 target_concept를 고르고,
socratic_q/hint/evaluate 노드가 이 내용을 쓴다.
"""

CONCEPTS = {
    "protein_balance": {
        "title": "탄수에 단백질 곁들이기",
        "criteria": "단백질은 하루 체중 1kg당 약 1.2~1.6g (criteria.txt)",
        "socratic_q": "이 끼니에 뭐가 좀 빠진 것 같아요?",
        "hints": [
            "영양소 종류를 떠올려봐요. 탄수 말고요.",
            "단백질 쪽은 어때요? (계란·두부·고기 같은)",
        ],
        "answer_keywords": ["단백질", "protein", "계란", "두부", "고기", "닭", "생선"],
        "praise": "정확해요! 계란 하나만 얹어도 균형이 살아요.",
    },
    "veggie_fiber": {
        "title": "채소·식이섬유 챙기기",
        "criteria": "배달·외식·야식엔 채소가 부족하기 쉽다 (criteria.txt)",
        "socratic_q": "색깔로 보면 빠진 게 있지 않아요?",
        "hints": [
            "접시에 초록색이 보이나요?",
            "채소·식이섬유 쪽이에요. (나물·샐러드·김치)",
        ],
        "answer_keywords": ["채소", "야채", "식이섬유", "나물", "샐러드", "김치", "녹색"],
        "praise": "맞아요! 나물 한 가지만 곁들여도 든든해져요.",
    },
    "hydration": {
        "title": "수분 챙기기",
        "criteria": "짠 음식·음주는 수분을 뺏고 수면의 질을 떨어뜨린다 (criteria.txt)",
        "socratic_q": "음식 말고 같이 챙기면 좋은 게 있을까요?",
        "hints": [
            "짠 걸 먹었다면 더 필요한 거예요.",
            "물이에요. 한 컵 곁들이면 좋아요.",
        ],
        "answer_keywords": ["물", "수분", "water", "음료", "차"],
        "praise": "그렇죠! 물 한 컵이면 충분해요.",
    },
}

# diagnose가 아직 안 배운 개념을 이 순서로 고른다
CONCEPT_ORDER = ["protein_balance", "veggie_fiber", "hydration"]


In [3]:
# 노드 8개 — evaluate/empathy/closing은 LLM, 나머지 규칙기반 (키 없으면 폴백)
"""그래프 노드 8개 (그래프설계.md §2).

각 노드 = state를 받아 바뀐 부분만 dict로 return 하는 함수.
지금은 **스텁** — LangGraph 강의 후 LLM 호출/판단 로직을 채운다.
(LLM 클라이언트는 llm.py로 분리 예정. 1주차는 규칙 기반으로 먼저 굴려보고 LLM은 그 다음.)
"""
import os

from dotenv import load_dotenv, find_dotenv


load_dotenv(find_dotenv(usecwd=True))  # OPENAI_API_KEY 로드 (cwd 기준, 노트북/스크립트 양쪽 안전)

# LLM (gpt-4o-mini). 키 없으면 None → 노드가 규칙기반으로 자동 폴백.
_llm = None
def get_llm():
    global _llm
    if _llm == "none":
        return None
    if _llm is None:
        if not os.getenv("OPENAI_API_KEY"):
            _llm = "none"; return None
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.4)
    return _llm

# 위험신호 키워드 (safety_guard) — 감지되면 코치 빠지고 전문가/1393
RISK_KEYWORDS = ["가슴이 아", "가슴 통증", "숨이 안", "죽고 싶", "자해", "쓰러"]


def safety_guard(state):
    """위험신호 감지 → risk_flag. (그래프설계 §2-0)"""
    text = state.get("today_input", "")
    risk = any(k in text for k in RISK_KEYWORDS)
    return {"risk_flag": risk, "stage": "safety_guard"}


def empathy(state):
    """criteria 톤으로 먼저 공감 (번역체 금지). (§2-1) — LLM, 키 없으면 규칙기반 폴백."""
    ti = state.get("today_input", "")
    llm = get_llm()
    if llm is None:
        return {"messages": [("ai", "오늘도 기록 남겨줘서 좋아요. 같이 천천히 살펴봐요.")],
                "stage": "empathy"}
    try:
        msg = llm.invoke(
            f'너는 따뜻한 셀프케어 코치야. 사용자가 오늘 이렇게 기록했어: "{ti}"\n'
            "판단·평가·지시 없이, 자연스러운 한국어 **친근한 존댓말(~요체, 반말 금지)**로 "
            "딱 한 문장만 공감해줘. 번역체·이모지 금지."
        ).content.strip()
    except Exception:
        msg = "오늘도 기록 남겨줘서 좋아요. 같이 천천히 살펴봐요."
    return {"messages": [("ai", msg)], "stage": "empathy"}


def diagnose(state):
    """아직 안 배운 개념 중 오늘 입력에 맞는 target_concept 선택. (§2-2)
    MVP: CONCEPT_ORDER에서 learned에 없는 첫 개념. (나중에 입력 맥락 매칭으로 고도화)
    """
    learned = state.get("learned", {})
    target = next((c for c in CONCEPT_ORDER if c not in learned), None)
    return {"target_concept": target, "stage": "diagnose"}


def socratic_q(state):
    """답 대신 질문. (§2-3)"""
    c = CONCEPTS[state["target_concept"]]
    return {"messages": [("ai", c["socratic_q"])], "stage": "socratic_q"}


def evaluate(state):
    """유저 답 채점 → verdict. (§2-4) — LLM 의미 판정, 키 없으면 키워드 매칭 폴백.
    (LLM은 '프로틴 쉐이크' 같은 키워드에 없는 표현도 개념적으로 맞으면 정답 처리)
    """
    c = CONCEPTS[state["target_concept"]]
    ans = state.get("user_answer", "")
    if not ans.strip():
        return {"verdict": "unknown", "stage": "evaluate"}
    llm = get_llm()
    if llm is None:  # 규칙기반 폴백
        v = "correct" if any(k in ans for k in c["answer_keywords"]) else "wrong"
        return {"verdict": v, "stage": "evaluate"}
    prompt = (
        f"개념: {c['title']} — {c['criteria']}\n"
        f"이 개념에서 맞다고 볼 답의 방향(예): {', '.join(c['answer_keywords'])}\n"
        f'사용자 답: "{ans}"\n'
        "사용자 답이 이 개념을 개념적으로 맞게 짚었으면 correct, 틀렸으면 wrong, "
        "모르겠다는 취지면 unknown. 다른 말 없이 딱 한 단어로만 답해."
    )
    try:
        out = llm.invoke(prompt).content.strip().lower()
    except Exception:
        out = ""
    v = "correct" if "correct" in out else ("unknown" if "unknown" in out else "wrong")
    return {"verdict": v, "stage": "evaluate"}


def praise(state):
    """맞음 → 칭찬 + learned 기록. (§2-5a)"""
    key = state["target_concept"]
    learned = {**state.get("learned", {}), key: 1}
    return {
        "learned": learned,
        "messages": [("ai", CONCEPTS[key]["praise"])],
        "stage": "praise",
    }


def hint(state):
    """틀림/모름 → 힌트 1개 + hint_count++. (§2-5b)"""
    n = state.get("hint_count", 0)
    c = CONCEPTS[state["target_concept"]]
    msg = c["hints"][min(n, len(c["hints"]) - 1)]
    return {"hint_count": n + 1, "messages": [("ai", msg)], "stage": "hint"}


def closing(state):
    """격려 + 다음에 스스로 해볼 것 1개. (§2-6) — LLM, 키 없으면 규칙기반 폴백."""
    learned = state.get("learned", {})
    llm = get_llm()
    if llm is None or not learned:
        if learned:
            last = CONCEPTS[list(learned)[-1]]["title"]
            msg = f"오늘 '{last}' 하나 스스로 찾아냈어요. 다음 끼니엔 뭐가 빠졌는지 먼저 떠올려봐요."
        else:
            msg = "오늘은 여기까지! 다음에 또 같이 살펴봐요."
        return {"messages": [("ai", msg)], "stage": "closing"}
    last = CONCEPTS[list(learned)[-1]]["title"]
    try:
        msg = llm.invoke(
            f"너는 셀프케어 코치야. 사용자가 오늘 '{last}' 개념을 스스로 깨쳤어. "
            "자연스러운 한국어 **친근한 존댓말(~요체, 반말 금지)**로, 칭찬 한 마디 + "
            "다음에 스스로 해볼 것 한 가지를 2문장으로. 번역체·이모지 금지."
        ).content.strip()
    except Exception:
        msg = f"오늘 '{last}' 하나 스스로 찾아냈어요. 다음 끼니엔 뭐가 빠졌는지 먼저 떠올려봐요."
    return {"messages": [("ai", msg)], "stage": "closing"}


# hint 루프 탈출 조건 (그래프설계 §3)
def hint_exhausted(state):
    return state.get("hint_count", 0) >= HINT_MAX


In [4]:
# 그래프 조립 — 분기·힌트 루프·유저 답 대기(interrupt_before)
"""LangGraph 그래프 조립 (그래프설계.md §3).

흐름: safety_guard →(위험? END : empathy)→ diagnose →(다 앎? closing : socratic_q)
      → [유저 답 대기] → evaluate →(correct: praise / else: hint)
      hint →(힌트 소진? closing : socratic_q 루프)  → praise → closing → END

핵심 3가지:
- 분기: add_conditional_edges (앎/모름, 채점 결과, 힌트 소진)
- 유저 답 대기: evaluate 앞에서 멈춤 = compile(interrupt_before=["evaluate"]) + checkpointer
- 힌트 루프: hint → socratic_q 로 되돌아가는 엣지 (HINT_MAX 넘으면 탈출)
"""
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver



def build_graph():
    b = StateGraph(CoachState)
    for fn in (safety_guard, empathy, diagnose, socratic_q, evaluate, praise, hint, closing):
        b.add_node(fn.__name__, fn)

    b.add_edge(START, "safety_guard")
    b.add_conditional_edges("safety_guard",
        lambda s: "risk" if s.get("risk_flag") else "ok",
        {"risk": END, "ok": "empathy"})            # 위험신호면 코치 빠지고 종료
    b.add_edge("empathy", "diagnose")
    b.add_conditional_edges("diagnose",
        lambda s: "done" if s.get("target_concept") is None else "teach",
        {"done": "closing", "teach": "socratic_q"})  # 분기① 다 배웠나/가르칠 게 있나
    b.add_edge("socratic_q", "evaluate")           # 이 사이에서 유저 답 대기(interrupt_before)
    b.add_conditional_edges("evaluate",
        lambda s: s["verdict"],
        {"correct": "praise", "wrong": "hint", "unknown": "hint"})  # 분기② 채점
    b.add_conditional_edges("hint",
        lambda s: "exhausted" if hint_exhausted(s) else "retry",
        {"retry": "socratic_q", "exhausted": "closing"})  # 힌트 루프 + 탈출
    b.add_edge("praise", "closing")
    b.add_edge("closing", END)

    # 유저 답을 기다리려면 evaluate 실행 직전에 멈춰야 함 → interrupt_before + checkpointer(재개용)
    return b.compile(checkpointer=MemorySaver(), interrupt_before=["evaluate"])


# ── 규칙기반 데모: LLM 없이 한 흐름 굴려보기 (라면+김밥 → 질문 → 답 → 힌트/칭찬) ──


In [5]:
# ── 데모: 한 흐름 굴려보기 (키 있으면 LLM, 없으면 규칙기반) ──
graph = build_graph()
cfg = {"configurable": {"thread_id": "demo"}}
seen = 0

def show():
    global seen
    msgs = graph.get_state(cfg).values.get("messages", [])
    for m in msgs[seen:]:
        print("   🩺", getattr(m, "content", m))
    seen = len(msgs)

print("[사용자] 점심에 라면이랑 김밥 먹었어")
graph.invoke({"today_input": "점심에 라면이랑 김밥 먹었어", "learned": {}, "hint_count": 0}, cfg)
show()  # empathy + 첫 소크라테스 질문까지 나오고 evaluate 앞에서 멈춤

for ans in ["글쎄 잘 모르겠는데", "아 단백질?"]:   # 첫 답 틀림→힌트, 둘째 맞음→칭찬
    print(f"[사용자] {ans}")
    graph.update_state(cfg, {"user_answer": ans})  # 유저 답 주입
    graph.invoke(None, cfg)                          # 멈춘 데서 재개
    show()

print("\n[진도 메모리]", graph.get_state(cfg).values.get("learned"))


[사용자] 점심에 라면이랑 김밥 먹었어
   🩺 점심에 라면이랑 김밥이라니, 정말 맛있었겠어요!
   🩺 이 끼니에 뭐가 좀 빠진 것 같아요?
[사용자] 글쎄 잘 모르겠는데
   🩺 영양소 종류를 떠올려봐요. 탄수 말고요.
   🩺 이 끼니에 뭐가 좀 빠진 것 같아요?
[사용자] 아 단백질?
   🩺 정확해요! 계란 하나만 얹어도 균형이 살아요.
   🩺 정말 대단하세요! 다음에는 탄수화물과 단백질을 조화롭게 섭취하는 레시피를 찾아보시면 좋을 것 같아요.

[진도 메모리] {'protein_balance': 1}


## 결과

`evaluate`(의미 판정)·`empathy`·`closing`은 **LLM(gpt-4o-mini)**로 판정/생성하고, 나머지는 규칙기반이에요. 키가 없으면 전부 규칙기반으로 자동 폴백해서 이 노트북은 **키 없이도 돌아갑니다.**

핵심: "프로틴 쉐이크"처럼 **키워드에 없는 답도 LLM이 단백질로 이해해 정답 처리**해요. (규칙기반이면 오답 처리)

**다음 단계**: `diagnose`를 입력 맥락에 맞게(치킨 → 채소 물어보기) · 의학 가드레일 실작동 · 진도 분기(다음 개념) · Streamlit 배포.